# UdaPlay — Notebook 00: Game Data Collection

This notebook dynamically builds the game dataset used by the UdaPlay agent.  
For each game title in `SEED_GAMES`, it:
1. Searches the web using the **Tavily API**
2. Passes the search results to **Claude** to extract structured game metadata
3. Saves the results to `data/games/games.json`

You can modify `SEED_GAMES` to collect data for **any** game.

## 1. Setup

In [ ]:
import sys
import os

# Add project root to path so src/ modules are importable
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

assert ANTHROPIC_API_KEY, 'ANTHROPIC_API_KEY not set in .env'
assert TAVILY_API_KEY, 'TAVILY_API_KEY not set in .env'
print('✓ API keys loaded')

In [ ]:
import anthropic
from tavily import TavilyClient

llm_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

print('✓ Clients initialized')

## 2. Configure Seed Games

Edit this list to collect data for any games you want.  
Re-running the collection cell will add new entries without duplicating existing ones.

In [ ]:
SEED_GAMES = [
    # Action
    "Elden Ring",
    "God of War Ragnarok",
    "Hades",
    "Sekiro Shadows Die Twice",
    "Devil May Cry 5",
    # RPG
    "The Witcher 3 Wild Hunt",
    "Baldur's Gate 3",
    "Cyberpunk 2077",
    "Persona 5 Royal",
    "Final Fantasy XVI",
    # Indie
    "Hollow Knight",
    "Stardew Valley",
    "Celeste",
    "Undertale",
    "Disco Elysium",
    # Sports / Other
    "FIFA 21",
    "Pokémon Scarlet and Violet",
    "The Legend of Zelda Breath of the Wild",
]

print(f'Seed list: {len(SEED_GAMES)} games')

## 3. Run Data Collection

In [ ]:
from src.data_collector import GameDataCollector
from tqdm.notebook import tqdm

OUTPUT_PATH = os.path.join(project_root, 'data', 'games', 'games.json')

collector = GameDataCollector(
    tavily_client=tavily_client,
    llm_client=llm_client,
    output_path=OUTPUT_PATH,
)

print(f'Output path: {OUTPUT_PATH}')
print(f'Collecting {len(SEED_GAMES)} games...')

In [ ]:
# Collect with progress bar
# delay_seconds=1.0 adds a 1s pause between API calls to avoid rate limits
all_games = collector.collect_all(SEED_GAMES, delay_seconds=1.0)

print(f'\n✓ Collection complete!')
print(f'  Successfully collected: {len(all_games)} games')
print(f'  Failed / skipped:       {len(collector.failed_titles)} titles')
if collector.failed_titles:
    print('  Failed titles:', collector.failed_titles)

## 4. Preview Collected Data

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame([
    {
        'Title': g['title'],
        'Developer': g['developer'],
        'Publisher': g['publisher'],
        'Release Date': g['release_date'],
        'Platforms': ', '.join(g['platforms']) if isinstance(g['platforms'], list) else g['platforms'],
        'Genre': ', '.join(g['genre']) if isinstance(g['genre'], list) else g['genre'],
        'Metacritic': g['metacritic_score'],
    }
    for g in all_games
])

print(f'Dataset shape: {df.shape}')
display(df.style.set_properties(**{'text-align': 'left'}))

## 5. Validate Schema Completeness

In [ ]:
from src.data_collector import REQUIRED_FIELDS

issues = []
for game in all_games:
    missing = [f for f in REQUIRED_FIELDS if f not in game or game[f] is None]
    if missing:
        issues.append({'title': game.get('title', '?'), 'missing_fields': missing})

if issues:
    print(f'⚠ {len(issues)} games have missing fields:')
    for issue in issues:
        print(f"  {issue['title']}: {issue['missing_fields']}")
else:
    print(f'✓ All {len(all_games)} games pass schema validation')

## 6. Add More Games (Incremental)

To add more games without re-collecting what's already saved, just add titles here and re-run.

In [ ]:
# Add any additional game titles here
ADDITIONAL_GAMES = [
    # "Red Dead Redemption 2",
    # "Minecraft",
]

if ADDITIONAL_GAMES:
    all_games = collector.collect_all(ADDITIONAL_GAMES, delay_seconds=1.0)
    print(f'Total games in dataset: {len(all_games)}')
else:
    print('No additional games to collect. Add titles to ADDITIONAL_GAMES above.')

## Summary

The dataset is saved to `data/games/games.json` and is ready for the RAG pipeline in **Notebook 01**.

| Field | Description |
|---|---|
| `game_id` | URL-safe slug identifier |
| `title` | Full game title |
| `developer` | Studio that built the game |
| `publisher` | Company that published it |
| `release_date` | ISO date (YYYY-MM-DD) |
| `platforms` | List of platforms |
| `genre` | List of genres |
| `description` | 2-4 sentence summary (embedded for RAG) |
| `source` | `"internal"` for collected data, `"web_search"` for agent-persisted data |